# Movie Agent

In [1]:
import json

import requests
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage

load_dotenv()

MODEL = "gpt-4o-mini"
MOVIE_API_BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
EXIT_COMMANDS = {"q", "quit", "quit", "exit"}

client = OpenAI()
messages = [
    {
        "role": "system",
        "content": (
            "You are a movie recommendation assistant. "
            "Use the movie API tools when the user asks about popular movies, "
            "movie details, cast, or crew. Keep answers concise and useful."
        ),
    }
]

In [2]:
def fetch_json(path: str):
    response = requests.get(f"{MOVIE_API_BASE_URL}{path}", timeout=20)
    response.raise_for_status()
    return response.json()


def get_popular_movies():
    movies = fetch_json("/movies")
    return json.dumps(
        [
            {
                "id": movie["id"],
                "title": movie["title"],
                "overview": movie.get("overview", ""),
                "release_date": movie.get("release_date"),
                "vote_average": movie.get("vote_average"),
            }
            for movie in movies[:10]
        ],
        ensure_ascii=False,
    )


def get_movie_details(id):
    movie = fetch_json(f"/movies/{id}")
    return json.dumps(
        {
            "id": movie["id"],
            "title": movie["title"],
            "overview": movie.get("overview", ""),
            "genres": [genre["name"] for genre in movie.get("genres", [])],
            "runtime": movie.get("runtime"),
            "release_date": movie.get("release_date"),
            "vote_average": movie.get("vote_average"),
        },
        ensure_ascii=False,
    )


def get_movie_credits(id):
    credits = fetch_json(f"/movies/{id}/credits")
    cast = [
        {
            "name": person["name"],
            "character": person.get("character"),
        }
        for person in credits
        if person.get("known_for_department") == "Acting"
    ][:10]
    crew = [
        {
            "name": person["name"],
            "job": person.get("known_for_department"),
        }
        for person in credits
        if person.get("known_for_department") != "Acting"
    ][:10]

    return json.dumps(
        {
            "cast": cast,
            "crew": crew,
        },
        ensure_ascii=False,
    )

In [3]:
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get current popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information for a movie by id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get cast and crew information for a movie by id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [4]:
def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                parsed_arguments = json.loads(arguments)
            except json.JSONDecodeError:
                parsed_arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            try:
                result = function_to_run(**parsed_arguments)
            except requests.RequestException as exc:
                result = json.dumps(
                    {"error": f"Movie API request failed: {exc}"},
                    ensure_ascii=False,
                )
            except Exception as exc:
                result = json.dumps({"error": str(exc)}, ensure_ascii=False)

            print(
                f"Ran {function_name} with args {parsed_arguments} for a result of {result}"
            )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content or ""})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [5]:
print("Movie Agent를 시작합니다. 종료하려면 q, quit, quit, exit 를 입력하세요.")

while True:
    message = input("대화를 입력하세요: ").strip()

    if message.lower() in EXIT_COMMANDS:
        print("대화를 종료합니다.")
        break

    if not message:
        continue

    messages.append({"role": "user", "content": message})
    print(f"User: {message}")
    call_ai()

Movie Agent를 시작합니다. 종료하려면 q, quit, quit, exit 를 입력하세요.
User: 추천 영화 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a result of [{"id": 1523145, "title": "Your Heart Will Be Broken", "overview": "High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but her family and classmates have reasons to separate the lovers.", "release_date": "2026-03-26", "vote_average": 6.531}, {"id": 83533, "title": "Avatar: Fire and Ash", "overview": "In the wake of the devastating war against the RDA and the loss of their eldest son, Jake Sully and Neytiri face a new threat on Pandora: the Ash People, a violent and power-hungry Na'vi tribe led by the ruthless Varang. Jake's family must fight for their survival and the future of Pandora in a conflict that pushes them to the